In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import mteb
from mteb.benchmarks import get_benchmark

In [3]:
VN_MTEB_BENCHMARK = 'VN-MTEB (vie, v1)'
vn_benchmark = mteb.benchmarks.get_benchmark(VN_MTEB_BENCHMARK)
all_tasks = list(vn_benchmark.tasks)

task_groups = {
    "STS":                [t for t in all_tasks if t.metadata.type == "STS"],
    "Retrieval":          [t for t in all_tasks if t.metadata.type == "Retrieval"],
    "Reranking":          [t for t in all_tasks if t.metadata.type == "Reranking"],
    "Classification":     [t for t in all_tasks if t.metadata.type == "Classification"],
    "Clustering":         [t for t in all_tasks if t.metadata.type == "Clustering"],
    "PairClassification": [t for t in all_tasks if t.metadata.type == "PairClassification"],
}

for name, tasks in task_groups.items():
    print(f"{name}: {len(tasks)} tasks")

STS: 3 tasks
Retrieval: 24 tasks
Reranking: 3 tasks
Classification: 12 tasks
Clustering: 5 tasks
PairClassification: 3 tasks


In [ ]:
import os
from sentence_transformers import SentenceTransformer

MODEL_NAMES = [
    'google/embeddinggemma-300m',
    os.path.abspath('../model/gemma300-vi-trimmed'),
]

# Choose which task group to run
RUN_TASKS = task_groups["STS"]  # change to "Retrieval", "Reranking", etc.

all_results = {}
for model_name in MODEL_NAMES:
    print(f"\n=== Evaluating: {model_name} ===")
    model = SentenceTransformer(model_name)
    all_results[model_name] = mteb.evaluate(
        model=model,
        tasks=RUN_TASKS,
        encode_kwargs={"batch_size": 16, 'show_progress_bar': True},
        show_progress_bar=True,
        num_proc=4,
        cache=mteb.cache.ResultCache("../vn_mteb_results"),
    )

In [6]:
# Compare results for both models grouped by task type
import pandas as pd

rows = []
for model_name, results in all_results.items():
    label = 'original' if 'gemma300-vi-trimmed-138' not in model_name else 'trimmed'
    df = results.to_dataframe()
    for _, row in df.iterrows():
        rows.append({'model': label, 'task': row['task_name'], 'score': row.iloc[-1]})

df = pd.DataFrame(rows).pivot(index='task', columns='model', values='score')
df['diff'] = df['trimmed'] - df['original']
print(df.sort_values('diff').to_string())

model            original   trimmed      diff
task                                         
STSBenchmark-VN  0.819424  0.783126 -0.036298
SICK-R-VN        0.788126  0.768511 -0.019615
BIOSSES-VN       0.783512  0.778217 -0.005295
